In [1]:
import pandas as pd
import os
import sys
sys.path.append("..")
import main

In [2]:
df = pd.DataFrame(columns=["Dataset", "Quality measure", "Beam width", "Number of subgroups", "Descriptions"])

for file in os.listdir("hyperparam_tuning_results/"):
    if not file.endswith(".csv"):
        continue
    file_name = file.replace(".csv", "")
    params = file_name.split("_")
    if file.startswith("brca"):
        params = ["brca_n"] + params[2:]
    elif file.startswith("polluted"):
        params = ["polluted_synthetic"] + params[2:]
    try:
        dataset = params[0]
        beam_size  = int(params[1][8:])
        measure = params[2]
        num_subgroups = int(params[-1][12:])
        if len(params) > 4:
            measure = "_".join(params[2:-1])
    except Exception as e:
        print(f"Could not parse file name: {file}")
        print(e)
        print(params)
        continue
    data = pd.read_csv(os.path.join("hyperparam_tuning_results/", file))
    descriptions = data["subgroup"].to_list()
    descriptions = [d.split(" AND ") for d in descriptions]
    df.loc[len(df)] = [dataset, measure, beam_size, num_subgroups, descriptions]

In [3]:
df.head()

,Dataset,Quality measure,Beam width,Number of subgroups,Descriptions
0,polluted_synthetic,chisquaredQF,100,3,"[[Attribute 1=='0', Attribute 5=='0', Attribut..."
1,polluted_synthetic,chisquaredQF,25,20,"[[Attribute 1=='0', Attribute 5=='0', Attribut..."
2,polluted_synthetic,chisquaredQF,100,20,"[[Attribute 1=='0', Attribute 5=='0', Attribut..."
3,synthetic,standardQF_0.5,250,50,"[[Attribute 4=='1', Attribute 5=='1'], [Attrib..."
4,polluted_synthetic,chisquaredQF,25,10,"[[Attribute 1=='0', Attribute 5=='0', Attribut..."


In [4]:
main.params["split"] = True

datasets = {
    "Facebook" : main.get_data("Facebook")[1],
    "brca_n" : main.get_data("brca_n")[1],
    "synthetic" : main.get_data("synthetic")[1],
    "polluted_synthetic" : main.get_data("polluted_synthetic")[1]
}

targets = {
    "Facebook" : main.get_target("Facebook"),
    "brca_n" : main.get_target("brca_n"),
    "synthetic" : main.get_target("synthetic"),
    "polluted_synthetic" : main.get_target("polluted_synthetic")
}

target_entries = {d: datasets[d][targets[d][0]] == targets[d][1] for d in datasets}



In [6]:
df_with_youden = df.copy()
for index, row in df.iterrows():
    global_entry = pd.Series(False, index = datasets[row["Dataset"]].index)
    for subgroup in row["Descriptions"]:
        entry = pd.Series(True, index = datasets[row["Dataset"]].index)
        for selector in subgroup:
            attr, val = selector.split("==")
            val = val.strip().strip("'").strip('"')
            entry = entry & (datasets[row["Dataset"]][attr] == val)
        global_entry = global_entry | entry
    positive = sum(global_entry & target_entries[row["Dataset"]]) / sum(target_entries[row["Dataset"]])
    negative = sum(global_entry & ~target_entries[row["Dataset"]]) / sum(~target_entries[row["Dataset"]])
    youden = positive - negative
    df_with_youden.at[index, "Youden"] = youden
df = df_with_youden

In [7]:
selected = df.groupby(["Dataset"]).apply(lambda x: x.loc[x["Youden"].idxmax()])
selected.drop(columns=["Descriptions"], inplace=True)
display(selected)
selected.to_csv("selected_hyperparams.csv", index=False)

/tmp/ipykernel_852804/3140843091.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected = df.groupby(["Dataset"]).apply(lambda x: x.loc[x["Youden"].idxmax()])


,Dataset,Quality measure,Beam width,Number of subgroups,Youden
Dataset,,,,,
Facebook,Facebook,standardQF_1.0,100,3,0.300537
brca_n,brca_n,standardQF_0.5,25,10,1.000000
polluted_synthetic,polluted_synthetic,standardQF_0.5,100,20,0.757925
synthetic,synthetic,standardQF_0.5,250,50,1.000000


In [9]:
print(selected.to_markdown())

| Dataset            | Dataset            | Quality measure   |   Beam width |   Number of subgroups |   Youden |
|:-------------------|:-------------------|:------------------|-------------:|----------------------:|---------:|
| Facebook           | Facebook           | standardQF_1.0    |          100 |                     3 | 0.300537 |
| brca_n             | brca_n             | standardQF_0.5    |           25 |                    10 | 1        |
| polluted_synthetic | polluted_synthetic | standardQF_0.5    |          100 |                    20 | 0.757925 |
| synthetic          | synthetic          | standardQF_0.5    |          250 |                    50 | 1        |
